# 09 Test Inference and Adapter Specialization

This notebook sends the same style of request to each adapter and compares outputs.

```mermaid
flowchart TB
    A[Prompt set] --> B[finance adapter]
    A --> C[legal adapter]
    A --> D[healthcare adapter]
    B --> E[Compare tone and terminology]
    C --> E
    D --> E
```

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Example prompts and expected behavior

- Finance should mention financial risk, margin, cash, revenue, or accounting context.
- Legal should mention contracts, clauses, liability, or counsel.
- Healthcare should mention coverage, care plans, discharge, or patient administration.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{cfg.vllm_base_url}/v1", api_key=cfg.vllm_api_key)
prompts = {
    "finance": "Explain revenue concentration risk in two sentences.",
    "legal": "Explain why a limitation of liability clause matters.",
    "healthcare": "Explain what prior authorization means.",
}

outputs = {}
for adapter, prompt in prompts.items():
    response = client.chat.completions.create(
        model=adapter,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=180,
    )
    outputs[adapter] = response.choices[0].message.content
    print(f"\n## {adapter}\nPrompt: {prompt}\nOutput: {outputs[adapter]}")

## Lightweight evaluation

Expected output: keyword scores logged to MLflow.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "evaluation/evaluate.py"], check=True)